# Week 1: Python & NumPy Mastery
## Notebook 12: Singular Value Decomposition (SVD) & Low-Rank Approximation

### 1. Conceptual Intuition & Theoretical Background
**Singular Value Decomposition (SVD)** factorizes any real $m \times n$ matrix $A$ into three constituent transformation matrices:
$$A = U \Sigma V^T$$
Where:
- $U \in \mathbb{R}^{m \times m}$: Orthonormal eigenvectors of $A A^T$ (Left singular vectors).
- $\Sigma \in \mathbb{R}^{m \times n}$: Diagonal matrix of non-negative singular values $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r \ge 0$.
- $V \in \mathbb{R}^{n \times n}$: Orthonormal eigenvectors of $A^T A$ (Right singular vectors).

**Eckart-Young-Mirsky Theorem:**
The optimal rank-$k$ approximation ($k < r$) minimizing Frobenius reconstruction error is obtained by truncating to the top $k$ singular values:
$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T, \quad \min_{\text{rank}(B)=k} \|A - B\|_F = \sqrt{\sum_{i=k+1}^{r} \sigma_i^2}$$

---

### 2. Scratch Python Implementation: SVD & Low-Rank Compression

In [ ]:
import numpy as np

def svd_from_scratch(A):
    """
    Computes SVD via eigendecomposition of A^T A and A A^T.
    A = U Sigma V^T
    """
    m, n = A.shape
    
    # 1. Right singular vectors V from A^T A
    AtA = np.dot(A.T, A)
    eig_vals_v, V = np.linalg.eigh(AtA)
    
    # Sort eigenvalues and eigenvectors descending
    sort_idx_v = np.argsort(eig_vals_v)[::-1]
    eig_vals_v = eig_vals_v[sort_idx_v]
    V = V[:, sort_idx_v]
    
    # Singular values are square roots of eigenvalues
    # Clamp negative floating point zeros
    singular_values = np.sqrt(np.maximum(eig_vals_v, 0.0))
    
    # 2. Left singular vectors U: u_i = A v_i / sigma_i
    U = np.zeros((m, len(singular_values)))
    for i in range(len(singular_values)):
        if singular_values[i] > 1e-10:
            U[:, i] = np.dot(A, V[:, i]) / singular_values[i]
        else:
            U[:, i] = 0.0
            
    return U, singular_values, V.T

# Low-Rank Matrix Compression Demo
np.random.seed(42)
rows, cols = 80, 60
# Create low-rank underlying signal + small Gaussian noise
true_signal = np.dot(np.random.randn(rows, 3), np.random.randn(3, cols))
A_matrix = true_signal + 0.05 * np.random.randn(rows, cols)

U, S, Vt = svd_from_scratch(A_matrix)

# Reconstruct with Rank-3 Truncation
k = 3
A_k = np.dot(U[:, :k] * S[:k], Vt[:k, :])

energy_retained = np.sum(S[:k]**2) / np.sum(S**2) * 100
reconstruction_error = np.linalg.norm(A_matrix - A_k, 'fro')

print("--- SVD Factorization & Truncation ---")
print(f"Original Matrix Dimensions:   {A_matrix.shape}")
print(f"Top 5 Singular Values:        {np.round(S[:5], 2)}")
print(f"Rank-{k} Energy Retained:       {energy_retained:.2f}%")
print(f"Frobenius Reconstruction Error: {reconstruction_error:.4f}")

### 3. Step-by-Step Code Walkthrough
1. **$A^T A$ Eigendecomposition:** Solves for orthogonal columns of $V$.
2. **Singular Values:** Computed as $\sigma_i = \sqrt{\lambda_i}$.
3. **Left Vectors ($U$):** Projected via $u_i = \frac{A v_i}{\sigma_i}$ guaranteeing orthonormality.
4. **Energy Retained:** Sum of squared top singular values reflects cumulative variance preserved.

---

### 4. Industry Applications & Trade-offs
- **Latent Semantic Analysis (LSA) & LoRA:** LoRA decomposes $\Delta W = B \cdot A$ using the same low-rank intuition established by SVD.
- **Trade-off:** Full SVD takes $O(\min(m^2 n, m n^2))$, requiring randomized SVD (Halko et al.) for billion-element tensors.

---

### 5. Top Interview Questions & Answers
**Q1: What is the relationship between SVD and Principal Component Analysis (PCA)?**
> **Answer:** If data matrix $X$ is zero-mean centered, the right singular vectors $V$ from SVD of $X$ are identical to the principal component direction eigenvectors of the covariance matrix $\frac{1}{n} X^T X$.

**Q2: How does SVD handle non-square or singular matrices?**
> **Answer:** Unlike standard eigendecomposition (which requires square diagonalizable matrices), SVD exists and is well-defined for any arbitrary $m \times n$ rectangular matrix.